# Phase 2 — Analyse Votes & Map Consensus

**What this does:** Loads vote responses (from a shared spreadsheet or form export),
clusters participants by their agreement patterns, and shows:
- A 2D opinion map (who's where)
- Which statements bridge all groups (group-informed consensus)
- Which statements are divisive (where facilitation energy is most needed)

---

## Expected votes CSV format

The simplest way to collect votes is a **shared Google Sheet** or **Excel Online file**:

| Name | Statement 1 | Statement 2 | Statement 3 | ... |
|------|-------------|-------------|-------------|-----|
| Sarah | Agree | Pass | Disagree | ... |
| James | Pass | Agree | Agree | ... |

- Column headers must match the labels from `session_data/label_map.json` exactly
  (i.e. `Statement 1`, `Statement 2`, etc.)
- Valid vote values: `Agree`, `Pass`, `Disagree` (case-insensitive)
- Empty cells are treated as Pass

You can also use a Google Form export — the column headers will be the question
titles, which should be `Statement 1`, `Statement 2`, etc.

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
VOTES_CSV  = "example_votes.csv"        # ← your votes spreadsheet export
NAME_COL   = "Name"                     # ← column containing participant names
SESSION_DIR = "session_data"            # ← must match what 01_mediate.ipynb saved
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

session_dir = Path(SESSION_DIR)

# Load the label map produced by 01_mediate.ipynb
with open(session_dir / "label_map.json") as f:
    label_map = json.load(f)   # {"Statement 1": {"id": "...", "text": "..."}, ...}

with open(session_dir / "statements.json") as f:
    session_data = json.load(f)

print(f"Loaded {len(label_map)} statements from session_data/label_map.json")
for label, info in label_map.items():
    print(f"  {label}: {info['text'][:80]}{'...' if len(info['text']) > 80 else ''}")

## Step 1 — Load and preview votes

In [ ]:
votes_df = pd.read_csv(VOTES_CSV)
print(f"{len(votes_df)} voters loaded")
print(f"Columns: {list(votes_df.columns)}")
votes_df

In [ ]:
# Map text votes to numeric values
VOTE_MAP = {
    "agree"          :  1,
    "strongly agree" :  1,
    "yes"            :  1,
    "pass"           :  0,
    "neutral"        :  0,
    "abstain"        :  0,
    ""               :  0,
    "disagree"       : -1,
    "strongly disagree": -1,
    "no"             : -1,
}

def parse_vote(val) -> int:
    if pd.isna(val):
        return 0
    return VOTE_MAP.get(str(val).strip().lower(), 0)

# Check which statement columns are present in the CSV
stmt_cols = [col for col in label_map if col in votes_df.columns]
missing   = [col for col in label_map if col not in votes_df.columns]

if missing:
    print(f"⚠  Columns not found in CSV (will be skipped): {missing}")
print(f"✓  Statement columns found: {stmt_cols}")

# Quick summary of vote distributions
print()
for col in stmt_cols:
    counts = votes_df[col].value_counts().to_dict()
    print(f"  {col}: {dict(sorted(counts.items()))}")

## Step 2 — Build the opinion matrix and run clustering

In [ ]:
from uuid import UUID
from group_consensus.clustering.opinion_matrix import OpinionMatrix
from group_consensus.clustering.engine import ClusteringEngine
from group_consensus.clustering.consensus import ConsensusDetector
from group_consensus.models.types import Statement, StatementType, Vote, VoteValue

matrix = OpinionMatrix()

# Register all statements
stmt_objects = {}
for label, info in label_map.items():
    if label not in stmt_cols:
        continue
    s = Statement(
        id=UUID(info["id"]),
        text=info["text"],
        type=StatementType(info["type"]),
        session_id=session_data["session_id"],
    )
    stmt_objects[label] = s
    matrix.add_statement(s)

# Register all voters and load their votes
for _, row in votes_df.iterrows():
    name = str(row[NAME_COL]).strip()
    pid  = name.lower().replace(" ", "_")
    matrix.add_participant(pid)
    for label, stmt in stmt_objects.items():
        raw   = row.get(label, None)
        val   = parse_vote(raw)
        vote  = Vote(
            participant_id=pid,
            statement_id=stmt.id,
            value=VoteValue(val),
            session_id=session_data["session_id"],
        )
        matrix.add_vote(vote)

print(f"✓ Opinion matrix: {len(matrix.participant_ids)} participants × {len(matrix.statement_ids)} statements")
print(f"  Total votes recorded: {len(matrix)}")

In [ ]:
engine = ClusteringEngine(max_clusters=5)
clustering = engine.run(matrix)

print(f"Clusters found    : {clustering.num_clusters}")
print(f"Silhouette score  : {clustering.silhouette_score:.3f}  ",
      "(>0.5 = clear groups, 0.2–0.5 = moderate, <0.2 = no clear structure)")
print()
for c in clustering.clusters:
    # Map pid back to display name
    names = [
        str(votes_df.loc[votes_df[NAME_COL].str.lower().str.replace(" ", "_") == pid, NAME_COL].values[0])
        if pid in votes_df[NAME_COL].str.lower().str.replace(" ", "_").values else pid
        for pid in c.participant_ids
    ]
    print(f"  Cluster {c.id}: {', '.join(names)}")

## Step 3 — 2D opinion map

In [ ]:
# Build a dataframe for plotting
name_lookup = {
    str(row[NAME_COL]).strip().lower().replace(" ", "_"): str(row[NAME_COL]).strip()
    for _, row in votes_df.iterrows()
}

plot_rows = []
for pid, (x, y) in clustering.coordinates_2d.items():
    cluster_id = next(
        (c.id for c in clustering.clusters if pid in c.participant_ids), -1
    )
    plot_rows.append({
        "name"   : name_lookup.get(pid, pid),
        "x"      : x,
        "y"      : y,
        "cluster": f"Group {cluster_id + 1}",
    })

plot_df = pd.DataFrame(plot_rows)

fig = px.scatter(
    plot_df,
    x="x", y="y",
    color="cluster",
    text="name",
    title="Opinion map — each dot is a person, proximity = similar voting pattern",
    labels={"x": "Opinion dimension 1 (UMAP)", "y": "Opinion dimension 2 (UMAP)"},
    color_discrete_sequence=px.colors.qualitative.Set2,
    width=750, height=500,
)
fig.update_traces(textposition="top center", marker_size=12)
fig.update_layout(legend_title_text="Opinion group")
fig.show()

## Step 4 — Bridging and divisive statements

In [ ]:
detector = ConsensusDetector(min_cluster_approval=0.6)
consensus = detector.detect(list(stmt_objects.values()), clustering)

DIVIDER = "═" * 60

print(DIVIDER)
print(f"  BRIDGING STATEMENTS  (approved by all {clustering.num_clusters} groups)")
print(DIVIDER)
if consensus.bridging_statements:
    for s in consensus.bridging_statements:
        approvals = clustering.statement_approval_by_cluster.get(str(s.id), {})
        approval_str = "  ".join(f"G{k+1}: {v:.0%}" for k, v in sorted(approvals.items()))
        print(f"  \"{s.text}\"")
        print(f"  Approval by group → {approval_str}")
        print()
else:
    print("  None found at the 60% threshold — try lowering min_cluster_approval in CONFIG.")

print(DIVIDER)
print("  DIVISIVE STATEMENTS  (sharp split between groups)")
print(DIVIDER)
if consensus.divisive_statements:
    for s in consensus.divisive_statements:
        approvals = clustering.statement_approval_by_cluster.get(str(s.id), {})
        approval_str = "  ".join(f"G{k+1}: {v:.0%}" for k, v in sorted(approvals.items()))
        print(f"  \"{s.text}\"")
        print(f"  Approval by group → {approval_str}")
        print()
else:
    print("  No strongly divisive statements found.")

In [ ]:
consensus.to_file(session_dir / "consensus_result.json")
print(f"✓ Saved consensus_result.json  ({len(consensus.bridging_statements)} bridging, {len(consensus.divisive_statements)} divisive)")

## Step 5 — Statement approval heatmap

In [ ]:
# Build a heatmap: statements × groups, colour = approval rate
stmt_labels  = []
group_labels = [f"Group {c.id + 1}" for c in clustering.clusters]
heat_values  = []
hover_texts  = []

for label, stmt in stmt_objects.items():
    approvals = clustering.statement_approval_by_cluster.get(str(stmt.id), {})
    row_vals  = [approvals.get(c.id, 0.0) for c in clustering.clusters]
    row_hover = [f"{v:.0%}" for v in row_vals]
    # Wrap long statement text for the y-axis label
    short = stmt.text[:55] + "..." if len(stmt.text) > 55 else stmt.text
    stmt_labels.append(f"{label}: {short}")
    heat_values.append(row_vals)
    hover_texts.append(row_hover)

fig2 = go.Figure(data=go.Heatmap(
    z=heat_values,
    x=group_labels,
    y=stmt_labels,
    text=hover_texts,
    texttemplate="%{text}",
    colorscale="RdYlGn",
    zmin=0, zmax=1,
    colorbar_title="Approval",
))
fig2.update_layout(
    title="Approval rate per statement per opinion group",
    xaxis_title="Opinion group",
    yaxis_title="Statement",
    height=max(300, 80 * len(stmt_labels)),
    width=600,
    margin=dict(l=400),
)
fig2.show()

## Step 6 — Mediator brief

A plain-text summary you can copy into a facilitation briefing.

In [ ]:
print(f"Topic: {session_data['topic']}")
print(f"Participants: {len(matrix.participant_ids)}  |  Opinion groups: {clustering.num_clusters}")
print(f"Silhouette score: {clustering.silhouette_score:.3f}")
print()

for c in clustering.clusters:
    names = [
        name_lookup.get(pid, pid)
        for pid in c.participant_ids
    ]
    print(f"  Group {c.id + 1} ({len(names)} people): {', '.join(names)}")

print()
print("Agreed common ground (all groups ≥60% approval):")
if consensus.bridging_statements:
    for s in consensus.bridging_statements:
        print(f"  ✓ {s.text}")
else:
    print("  — none at 60% threshold")

print()
print("Where groups diverge most (worth exploring in facilitation):")
if consensus.divisive_statements:
    for s in consensus.divisive_statements[:3]:
        print(f"  ✗ {s.text}")
else:
    print("  — no strongly divisive statements found")